In [0]:
from pyspark.sql.functions import col, count, when, coalesce, lit

data = [(1, "Alice", "IT", "9998887777", None, 55000),
        (2, "Ben", "HR", None, None, 48000),
        (3, "Cara", None, None, "8887776666", 62000),
        (4, None, "Finance", "7776665555", None, None)]
columns = ["id", "name", "department", "mobile", "landline", "salary"]

customers = spark.createDataFrame(data, columns)
customers.display()

In [0]:
# Measure First - null count and percentage across every column
total = customers.count()

customers.select([count(when(col(c).isNull(), c)).alias(c) for c in customers.columns]).display()

customers.select([(count(when(col(c).isNull(), c))/total * 100).alias(c) for c in customers.columns]).display()



In [0]:
# dropna() - try all three variants and compare row counts
print("dropna() no args:", customers.dropna().count())
print("dropna(subset=['id']):",customers.dropna(subset=['id']).count())
print("dropna(thresh=4):", customers.dropna(thresh=4).count())
print("dropna(how='any'):", customers.dropna(how='any').count())
print("dropna(how='all'):", customers.dropna(how='all').count())


In [0]:
#Filling Nulls with fillna()
customers.display()
customers.fillna(0).display()
# fills ALL null numeric columns with 0 - risky if 0 has real business meaning

customers.fillna("Unknown", subset=['department']).display()
# fills only department, with an honest placeholder


customers.fillna({"department": "Unknown", "salary": 0}).display()
# different fill value per column - most realistic pattern

#Note: ⚠️ Filling a numeric column's nulls with 0 is one of the most common real mistakes in data cleaning — 0 often looks like a valid, meaningful value (zero salary, zero orders) rather than “this was actually missing.” This can quietly corrupt downstream aggregations (Day 6) like averages. Prefer a clearly-impossible sentinel value or, better, keep it null and handle it explicitly in your checks.




In [0]:
#coalesce() — the smarter fallback for mobile/landline
from pyspark.sql.functions import coalesce
customers_contact = customers.withColumn(
    "contact", coalesce(col("mobile"), col("landline"), lit("Not Available"))
)
customers_contact.select("name", "mobile", "landline", "contact").display()
# coalesce() takes a list of columns and returns the first non-null value

#Note: ⚠️ coalesce() is not a replacement for fillna()! It's a tool for combining columns, not filling them. coalesce() is a function that takes a list of columns and returns the first non-null value. fillna() is a method that takes a value and fills all nulls with that value. coalesce() is useful for combining columns, but it's not a replacement for fillna(). fillna() is useful for filling nulls, but it's not a replacement for coalesce()

In [0]:
#Put it together — one deliberate strategy per column
customers_clean = (customers
    .dropna(subset=["id"])
    .withColumn("contact", coalesce(col("mobile"), col("landline"), lit("Not Available")))
    .fillna({"department": "Unknown"})
)
customers_clean.display()